# PARA: Prompt-Anchored Residual Adaptation (beginner annotated)

Paper: Kang, Yue, Liu, Qin. *Prompt-Anchored Residual Adaptation for Biomedical Vision-Language Models* (arXiv 2609.33701, 2026).

Every code line has a `# comment`. Read the markdown above each cell, then run it with Shift+Enter.

**Runtime:** `Runtime > Change runtime type > T4 GPU`. The single few-shot run takes about 5 minutes;
the optional repeated-support protocol (section 11) adds roughly 2 minutes per support draw.

**Note:** no official code was available, so this notebook follows the equations in the paper. Wherever the
paper leaves a detail open, the cell says what was chosen and why (look for **"Not specified in the paper"**).

## 0. The idea in 3 minutes

**Background.** BiomedCLIP is a CLIP-style model trained on 15M biomedical image-caption pairs. It can classify
an image *zero-shot*: embed the image, embed a text description of each class, pick the class whose text is
most similar. That frozen text-based prediction is the **prompt prediction**.

**The problem.** With a few labelled images per class (*few-shot*, the **support set**), we can adapt the model
(CoOp learns prompts, CLIP-LoRA tunes the image encoder, ...). But a handful of images only covers part of each
class. If you happen to draw "bad" examples, the adapted model can end up *worse* than zero-shot. The paper shows
CoOp on COVID-19 ranging from 33.9% to 74.4% accuracy depending only on *which* 20 support sets were drawn.

**PARA's answer: never throw the frozen prediction away; move away from it by a measured step.**

```
            frozen text prediction (the ANCHOR, same for every support set)
                 |
 final score  =  z0  +  eta(S) * ( zS  -  z0 )
                                    |
                    adapted visual prediction (learned from the support set S)
```

- **z0 (anchor branch):** frozen BiomedCLIP image embedding vs frozen text anchors. Never trained.
- **zS (visual branch):** image encoder with small trainable **LoRA** adapters, compared with **class prototypes**
  = text anchor + sum of that class's adapted support embeddings.
- **eta(S) (step size), in [0, lambda):** computed *in closed form* (no training) from the support set in the
  frozen embedding space using two quantities per class:
  - **anchor discrepancy** delta_c: how far the support images' mean direction is from the text anchor.
    Large = the text is missing something the images show.
  - **support agreement** phi_c: how much the support images agree with each other.
    Large = the evidence is consistent, so it is safer to trust it.

  A class pushes eta up only when **both** are large: the images disagree with the text **and** agree with each other.

**Base-to-novel extra:** classes with no labelled images (novel) can't have a visual branch. PARA instead nudges each
novel class's text anchor by a weighted average of the "image minus text" shifts seen on similar base classes.

### What this notebook builds

| Section | Paper | What |
|---|---|---|
| 5 | Eq. 1 | text anchors from a prompt ensemble |
| 6 | Eq. 8 | zero-shot anchor branch |
| 7 | (Hu et al.) | LoRA adapters on q, k, v, output projections of every attention layer |
| 8 | Eq. 2-3 | leave-one-out prototype training |
| 9 | Eq. 4-7, 9-10 | visual branch, support geometry, eta(S), final PARA prediction |
| 10 | Table 4 | free ablations: visual only, fixed eta = 0.5 |
| 11 | Appx A.2 | repeated-support protocol: mean and CVaR20 over several support draws |
| 12 | Eq. 11-14 | base-to-novel residual transport |

## 1. Setup

- `open_clip_torch` loads **BiomedCLIP** from the Hugging Face hub (the backbone used in the paper).
- `medmnist` gives small, standardised biomedical datasets with one-line download. **DermaMNIST** (skin lesion
  dermoscopy, 7 classes) is one of the paper's 11 datasets, so we use it by default.

In [ ]:
!pip -q install open_clip_torch medmnist

In [ ]:
import os, math, time, random, contextlib       # standard library helpers
from glob import glob                            # find files by pattern (own-dataset option)

import numpy as np                               # arrays and math on the CPU
import torch                                     # PyTorch, the deep learning library
import torch.nn as nn                            # building blocks for neural networks (layers, Parameter)
import torch.nn.functional as F                  # functions like cross_entropy and normalize
import matplotlib.pyplot as plt                  # plotting
from PIL import Image                            # open / convert images
from torchvision import transforms as Tr         # image augmentations (random crop, flip)
from sklearn.metrics import f1_score             # Macro-F1, the paper's second metric
import open_clip                                 # loads BiomedCLIP and its tokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"   # use the GPU if there is one
USE_AMP = device == "cuda"                       # mixed precision (16-bit) training only makes sense on a GPU
print("Device:", device)

def set_seed(seed):
    # Fix every random number generator so a run can be repeated exactly
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

## 2. Settings

All knobs in one place. Values marked *(paper)* are the ones reported in the paper's implementation details.

In [ ]:
SEED = 1                   # master random seed

DATASET = "dermamnist"     # any 2D single-label MedMNIST name ("dermamnist", "octmnist", "pathmnist",
                           # "bloodmnist", "breastmnist", ...) or "folder" for your own images (see section 4)
MEDMNIST_SIZE = 224        # MedMNIST+ image size: 224 matches BiomedCLIP's input; 128 or 64 download faster
DATA_DIR = "/content/my_dataset"   # only used when DATASET = "folder"
N_TEST = 2000              # max test images (random subset) to keep evaluation quick; None = all

SHOTS = 16                 # K: labelled support images per class (paper: K in {1, 2, 4, 8, 16})

LORA_RANK = 8              # rank r of each LoRA update (paper)
LORA_ALPHA = 16            # LoRA scaling alpha; the update is multiplied by alpha / r = 2 (paper)
EPOCHS = 80                # passes over the support set (paper)
BATCH_SIZE = 32            # support images per training step (not specified in the paper)
LR_LORA = 1e-4             # AdamW learning rate for the LoRA weights (paper)
LR_SCALE = 1e-3            # AdamW learning rate for the visual logit scale tau_v (paper)
WEIGHT_DECAY = 1e-4        # AdamW weight decay (paper)

LAMBDA = 0.9               # global upper bound of the residual step eta(S) (paper, same for all datasets)
BETA, GAMMA = 20.0, 0.2    # base-to-novel transport: semantic selectivity beta, transfer size gamma (paper)

set_seed(SEED)             # make everything below repeatable

## 3. Load BiomedCLIP and freeze it

BiomedCLIP = a ViT-B/16 image encoder + a PubMedBERT text encoder, both mapping into a shared 512-number space.

- **Freezing:** `requires_grad_(False)` means PyTorch will never change these weights. The only trainable things
  will be the LoRA adapters we insert later, plus one number (`tau_v`).
- **`tau0` (logit scale):** a learned "temperature" (about 100 for CLIP). Cosine similarities live in [-1, 1];
  multiplying by tau0 spreads them out so the softmax is sharp. The paper calls it the *frozen pretrained logit scale*.
- We keep the whole model in `eval()` mode all the time: ViT has no batch-norm, and BiomedCLIP's dropout is 0,
  so train/eval mode makes no difference for it.

In [ ]:
MODEL_ID = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"   # official BiomedCLIP checkpoint
model, preprocess = open_clip.create_model_from_pretrained(MODEL_ID)   # model + deterministic image preprocessing
tokenizer = open_clip.get_tokenizer(MODEL_ID)    # turns sentences into token IDs for PubMedBERT
model = model.to(device).eval()                  # move to GPU, evaluation mode
for p in model.parameters():                     # loop over every pretrained weight tensor
    p.requires_grad_(False)                      # freeze it

tau0 = model.logit_scale.exp().item()            # frozen logit scale; stored as a log value, so exp() it
print(f"tau0 = {tau0:.2f}")
print(preprocess)                                # resize to 224, centre crop, convert to tensor, normalise

## 4. Data: a support pool and a fixed test set

The paper splits every dataset once into a **support pool** (30%, where few-shot examples are drawn from) and a
fixed **test set** (70%). MedMNIST already comes with official splits, so we use its `train` split as the support
pool and its `test` split as the test set. They never overlap, which is what matters.

What this cell builds:
- `classnames`: list of class names
- `pool_labels`, `get_pool(i)`: labels of the support pool, and a function returning pool image `i` as a PIL image
- `test_idx`, `y_test`, `get_test(i)`: same for the test set

**Your own data (`DATASET = "folder"`):** one sub-folder per class, e.g. `my_dataset/benign/*.png`.
Each class is split 30 / 70 into pool / test like in the paper.

In [ ]:
if DATASET != "folder":
    import medmnist                                          # the MedMNIST package
    from medmnist import INFO                                # metadata: class names, task type, ...
    info = INFO[DATASET]                                     # info for the chosen dataset
    assert info["task"].startswith("multi-class") or info["task"] == "binary-class", "need a single-label dataset"
    DataClass = getattr(medmnist, info["python_class"])      # e.g. medmnist.DermaMNIST
    root = "/content/medmnist" if os.path.isdir("/content") else "./medmnist"   # where the .npz file goes
    os.makedirs(root, exist_ok=True)                         # create the folder if it doesn't exist
    pool_ds = DataClass(split="train", download=True, size=MEDMNIST_SIZE, root=root)   # support pool
    test_ds = DataClass(split="test", download=True, size=MEDMNIST_SIZE, root=root)    # test set
    classnames = [info["label"][str(i)] for i in range(len(info["label"]))]   # "0" -> "actinic keratoses ..."
    pool_labels = pool_ds.labels.flatten().astype(np.int64)  # stored as uint8 [N, 1]; flatten to [N], int64 for PyTorch
    test_labels_all = test_ds.labels.flatten().astype(np.int64)
    # "getter" functions: index -> PIL image. MedMNIST stores raw pixels; .convert("RGB") makes grayscale 3-channel
    get_pool = lambda i: Image.fromarray(pool_ds.imgs[i]).convert("RGB")
    get_test = lambda i: Image.fromarray(test_ds.imgs[i]).convert("RGB")

else:
    EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff")   # accepted image file types
    class_dirs = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
    classnames = [d.replace("_", " ") for d in class_dirs]   # folder "benign_tumor" -> class "benign tumor"
    rng_split = np.random.default_rng(SEED)                  # seeded RNG so the split is the same every run
    pool_paths, pool_y, test_paths, test_y = [], [], [], []
    for ci, d in enumerate(class_dirs):                      # ci = class number
        ps = sorted(p for p in glob(os.path.join(DATA_DIR, d, "*")) if p.lower().endswith(EXTS))
        ps = list(rng_split.permutation(ps))                 # shuffle this class's files
        cut = int(round(0.3 * len(ps)))                      # 30% support pool, 70% test (paper)
        pool_paths += ps[:cut]; pool_y += [ci] * cut
        test_paths += ps[cut:]; test_y += [ci] * (len(ps) - cut)
    pool_labels, test_labels_all = np.array(pool_y), np.array(test_y)
    get_pool = lambda i: Image.open(pool_paths[i]).convert("RGB")
    get_test = lambda i: Image.open(test_paths[i]).convert("RGB")

C = len(classnames)                                          # number of classes
rng = np.random.default_rng(SEED)                            # RNG for the test subset
test_idx = np.arange(len(test_labels_all))                   # all test indices
if N_TEST is not None and len(test_idx) > N_TEST:
    test_idx = np.sort(rng.choice(test_idx, N_TEST, replace=False))   # random subset, sorted for tidiness
y_test = test_labels_all[test_idx]                           # labels of the test images we use
y_test_t = torch.tensor(y_test, device=device)               # same, as a GPU tensor

print(f"{C} classes:", classnames)
print("support pool per class:", np.bincount(pool_labels, minlength=C))
print("test images per class :", np.bincount(y_test, minlength=C))

### Drawing a support set

A **support draw** = exactly `K` random pool images per class. The paper evaluates many draws (20) and gives every
method the *same* draws. Here a draw is identified by an integer seed, so draw `3` is always the same images.

*(The paper also balances how often each pool image is reused across draws. We just draw independently.)*

In [ ]:
def draw_support(k, draw_seed, classes=None):
    # Returns pool indices of k images per class (classes = all classes unless a subset is given)
    rng_d = np.random.default_rng(1000 + draw_seed)          # one fixed RNG per draw
    classes = range(C) if classes is None else classes
    idx = []
    for c in classes:
        cand = np.where(pool_labels == c)[0]                 # pool images of class c
        idx += list(rng_d.choice(cand, size=min(k, len(cand)), replace=False))   # k of them, no repeats
    return np.array(idx)

sup_idx = draw_support(SHOTS, draw_seed=0)                   # the draw we use for sections 8-10
print("support set size:", len(sup_idx), "| labels:", np.bincount(pool_labels[sup_idx], minlength=C))

# Look at one support image per class
fig, ax = plt.subplots(1, C, figsize=(2 * C, 2.4))
for c in range(C):
    i = sup_idx[pool_labels[sup_idx] == c][0]               # first support image of class c
    ax[c].imshow(get_pool(i)); ax[c].set_title(classnames[c][:18], fontsize=8); ax[c].axis("off")
plt.tight_layout(); plt.show()

## 5. Text anchors (Eq. 1)

Each class `c` gets a **prompt ensemble** of M sentences. Its **text anchor** is

$$t_c = \frac{\sum_m F_t(p_m(c))}{\lVert \sum_m F_t(p_m(c)) \rVert}$$

i.e. embed every sentence, normalise each to length 1, average, and normalise again. Averaging many phrasings
gives a more robust class direction than any single sentence.

The paper uses the **50 GPT-4 descriptions per class released with BiomedCoOp** ("Give N textual descriptions of
visual discriminative features for ... found in [MODALITY]"). We include 5 hand-written descriptions per class for
DermaMNIST in the same spirit. For other datasets we fall back to generic templates. To reproduce the paper exactly,
replace `DESCRIPTIONS` with BiomedCoOp's JSON files.

In [ ]:
MODALITY = {"dermamnist": "dermoscopy", "octmnist": "optical coherence tomography", "pathmnist": "histopathology",
            "bloodmnist": "blood cell microscopy", "breastmnist": "breast ultrasound", "pneumoniamnist": "chest X-ray",
            "retinamnist": "fundus photography", "organamnist": "abdominal CT", "tissuemnist": "kidney histology",
            }.get(DATASET, "medical")                      # imaging modality, used inside the prompts

# Visual descriptions per class (DermaMNIST). Keys must match the class names exactly.
DESCRIPTIONS = {
    "actinic keratoses and intraepithelial carcinoma": [
        "a rough scaly patch with a red strawberry pattern around hair follicles",
        "white to yellow surface scale over a pink background",
        "rosettes and target-like follicular openings",
        "fine wavy vessels on a pink-red lesion",
        "a flat or slightly raised lesion on sun-damaged skin"],
    "basal cell carcinoma": [
        "arborizing tree-like branching vessels",
        "blue-gray ovoid nests and globules",
        "leaf-like pigmented areas at the periphery",
        "shiny white-red structureless areas",
        "ulceration without a pigment network"],
    "benign keratosis-like lesions": [
        "milia-like cysts and comedo-like openings",
        "a cerebriform brain-like surface pattern",
        "a sharply demarcated waxy stuck-on appearance",
        "fingerprint-like light brown structures",
        "moth-eaten borders"],
    "dermatofibroma": [
        "a central white scar-like patch",
        "a delicate light brown pigment network at the periphery",
        "a small firm brown nodule",
        "ring-like globules around a white center",
        "a symmetric lesion with a light center and darker rim"],
    "melanoma": [
        "an asymmetric lesion with irregular borders",
        "multiple colors including black, brown, blue and red",
        "an atypical pigment network with thick irregular lines",
        "a blue-white veil",
        "irregular dots, globules and streaks at the periphery"],
    "melanocytic nevi": [
        "a symmetric lesion with a regular pigment network",
        "a uniform brown color",
        "a smooth regular border",
        "evenly distributed dots and globules",
        "a small round homogeneous lesion"],
    "vascular lesions": [
        "red, purple or blue lacunae",
        "well-demarcated red-blue lobules",
        "whitish fibrous septa separating red areas",
        "no pigment network",
        "a homogeneous bright red to dark purple color"],
}

TEMPLATES = ["a {m} image of {c}.", "a {m} image showing {c}.", "{c} as seen in {m}.",
             "an example of {c} in a {m} image.", "a medical image of {c}."]   # fallback prompts

def class_prompts(name):
    # Build the prompt ensemble {p_m(c)} for one class
    if name in DESCRIPTIONS:                                # descriptive prompts when available
        return [f"a {MODALITY} image of {name}, showing {d}." for d in DESCRIPTIONS[name]]
    return [t.format(m=MODALITY, c=name) for t in TEMPLATES]   # otherwise generic templates

@torch.no_grad()                                            # text encoder is frozen: no gradients
def text_anchor(prompts):
    tok = tokenizer(prompts, context_length=256).to(device)  # [M, 256] token IDs (PubMedBERT max length 256)
    e = F.normalize(model.encode_text(tok).float(), dim=-1)  # [M, 512], each sentence embedding has length 1
    return F.normalize(e.mean(0), dim=-1)                   # average, then normalise again (Eq. 1) -> [512]

T_anchor = torch.stack([text_anchor(class_prompts(c)) for c in classnames])   # [C, 512]: one t_c per class
print("text anchors:", tuple(T_anchor.shape))
print("example prompt:", class_prompts(classnames[0])[0])

## 6. The anchor branch: zero-shot BiomedCLIP (Eq. 8)

$$z^0_c(x) = \tau_0 \langle F^0_v(x), t_c \rangle$$

Frozen image embedding (`F0_v`) dotted with each text anchor. Since both have length 1, the dot product is the cosine
similarity. This prediction **does not depend on the support set at all**, which is why the paper calls it
*support-invariant*.

We encode the test images once now with the frozen encoder and reuse them everywhere (the anchor branch and the
base-to-novel section both need them).

`encode(...)` takes an `adapted` flag: later, when LoRA is inserted, `adapted=False` switches it off temporarily so we
get the original frozen encoder `F0_v`, and `adapted=True` gives the adapted encoder `FS_v`.

In [ ]:
lora_modules = []                                    # filled in section 7; empty = encoder is plain BiomedCLIP

@contextlib.contextmanager
def lora_off():
    # Temporarily disable every LoRA adapter -> the image encoder becomes the frozen F0_v again
    for m in lora_modules: m.enabled = False
    try:
        yield
    finally:
        for m in lora_modules: m.enabled = True     # always switch back on, even if an error happened

@torch.no_grad()                                     # encoding only, no training here
def encode(getter, indices, adapted, bs=128):
    # getter(i) -> PIL image; returns [N, 512] unit-length image embeddings
    feats = []
    ctx = contextlib.nullcontext() if adapted else lora_off()   # choose F^S_v (adapted) or F^0_v (frozen)
    with ctx:
        for i in range(0, len(indices), bs):         # batches of 128 images
            x = torch.stack([preprocess(getter(j)) for j in indices[i:i + bs]]).to(device)   # [b, 3, 224, 224]
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):   # 16-bit math on GPU = faster
                f = model.encode_image(x)            # [b, 512]
            feats.append(F.normalize(f.float(), dim=-1))   # back to 32-bit, length 1
    return torch.cat(feats)                          # [N, 512]

def standardize(z):
    # Per-image z-score across classes: (score - mean over classes) / std over classes.
    # Removes the effect of each branch's own logit scale so the two branches can be mixed fairly.
    return (z - z.mean(-1, keepdim=True)) / (z.std(-1, keepdim=True, unbiased=False) + 1e-8)

def metrics(logits, y):
    # Accuracy and Macro-F1 (mean of per-class F1) in %, the paper's two metrics
    pred = logits.argmax(-1).cpu().numpy()          # predicted class = highest score
    y = y.cpu().numpy() if torch.is_tensor(y) else y
    acc = (pred == y).mean() * 100
    f1 = f1_score(y, pred, average="macro", labels=np.unique(y), zero_division=0) * 100
    return acc, f1

t0 = time.time()
f0_test = encode(get_test, test_idx, adapted=False)  # [N_test, 512] frozen test embeddings F0_v(x)
z0_test = tau0 * f0_test @ T_anchor.T                # [N_test, C] anchor scores z0 (Eq. 8)
print(f"encoded {len(test_idx)} test images in {time.time() - t0:.1f}s")
acc, f1 = metrics(z0_test, y_test)
print(f"Zero-shot BiomedCLIP (anchor branch) | Acc {acc:.2f} | Macro-F1 {f1:.2f}")

## 7. LoRA: a small trainable update inside the image encoder

**LoRA (Low-Rank Adaptation)** leaves a frozen weight matrix `W` untouched and learns a small correction:

$$y = Wx + b + \frac{\alpha}{r} B A x, \qquad A \in \mathbb{R}^{r \times d_{in}},\; B \in \mathbb{R}^{d_{out} \times r}$$

With rank r = 8 and d = 768, `A` and `B` have 2 x 8 x 768 = 12k numbers, versus 590k in `W`.
`B` starts at **zero**, so at the start the adapted encoder is *exactly* the frozen one.

The paper puts LoRA on the **query, key, value and output projections** of every attention layer (12 layers).
In timm's ViT, q, k, v are computed by one fused layer `qkv` (768 -> 3 x 768), so we give that layer **three**
separate (A, B) pairs, one per q / k / v slice. This is the same as having separate LoRA on q, k and v.

Also trained: `tau_v`, the **visual logit scale** (one number), created in section 8.

In [ ]:
class LoRALinear(nn.Module):
    # Wraps a frozen nn.Linear and adds (alpha / r) * B A x.
    # n_chunks > 1 splits the output into equal slices with their own (A, B), e.g. 3 for fused q|k|v.
    def __init__(self, base, r, alpha, n_chunks=1):
        super().__init__()                                   # required first line for nn.Module subclasses
        self.base = base                                     # the original frozen layer
        self.scale = alpha / r                               # LoRA scaling factor (16 / 8 = 2)
        dev = base.weight.device                             # create new weights on the same device
        out_chunk = base.out_features // n_chunks            # output size of each slice (768 for q, k or v)
        self.A = nn.ParameterList([nn.Parameter(torch.empty(r, base.in_features, device=dev)) for _ in range(n_chunks)])
        self.B = nn.ParameterList([nn.Parameter(torch.empty(out_chunk, r, device=dev)) for _ in range(n_chunks)])
        self.enabled = True                                  # lora_off() flips this to False
        self.reset()                                         # initialise A and B

    def reset(self):
        # Fresh start: A random (standard LoRA init), B zero -> the update B A x is exactly 0
        for A in self.A: nn.init.kaiming_uniform_(A, a=math.sqrt(5))
        for B in self.B: nn.init.zeros_(B)

    def forward(self, x):
        y = self.base(x)                                     # frozen W x + b
        if not self.enabled:                                 # adapters switched off -> frozen encoder
            return y
        # x @ A^T -> [.., r], then @ B^T -> [.., out_chunk]; concatenate the slices -> [.., out_features]
        delta = torch.cat([F.linear(F.linear(x, A.to(x.dtype)), B.to(x.dtype)) for A, B in zip(self.A, self.B)], dim=-1)
        return y + self.scale * delta                        # frozen output + low-rank correction

# Insert LoRA into every attention layer of the ViT image encoder
for blk in model.visual.trunk.blocks:                        # 12 transformer blocks
    attn = blk.attn                                          # the self-attention module of this block
    attn.qkv = LoRALinear(attn.qkv, LORA_RANK, LORA_ALPHA, n_chunks=3)   # q, k, v projections
    attn.proj = LoRALinear(attn.proj, LORA_RANK, LORA_ALPHA)             # output projection
    lora_modules += [attn.qkv, attn.proj]                    # remember them (for lora_off and reset)

lora_params = [p for m in lora_modules for p in list(m.A) + list(m.B)]   # every trainable LoRA tensor
print(f"LoRA modules: {len(lora_modules)} | trainable numbers: {sum(p.numel() for p in lora_params):,}"
      f" | frozen BiomedCLIP image encoder: {sum(p.numel() for p in model.visual.parameters() if not p.requires_grad):,}")

# Sanity check: B = 0, so the adapted encoder must equal the frozen one right now
chk = encode(get_test, test_idx[:4], adapted=True)
print("max |adapted - frozen| at init:", (chk - f0_test[:4]).abs().max().item())   # ~0 (tiny fp16 noise)

## 8. Training the visual branch: leave-one-out prototypes (Eq. 2-3)

**Class prototype (Eq. 2):** text anchor + adapted embeddings of that class's support images, normalised:

$$p^S_c = \frac{t_c + \sum_{i \in S_c} F^S_v(x_i)}{\lVert t_c + \sum_{i \in S_c} F^S_v(x_i) \rVert}$$

Because `t_c` is in the sum, the prototype is **anchored** to the text: with few images it stays close to the text,
with many images the images dominate.

**Why leave-one-out?** If image `x_i` is part of its own class prototype, the easy way to lower the loss is to make
`x_i` similar to *itself*, not to its class. So when classifying training image `i`, its own embedding is removed
from its own class prototype (other classes' prototypes stay as they are):

$$\mathcal{L}_{vis} = \frac{1}{N}\sum_i \mathrm{CE}\Big(\tau_v \big[\langle F^S_v(x_i), p^S_{c,-i}\rangle\big]_{c}, y_i\Big)$$

With 1 shot, the positive prototype is just the text anchor `t_c`, and the image still trains the adapter.

**Implementation detail (Not specified in the paper):** the prototypes need the adapted embedding of *every* support image,
but we only push `BATCH_SIZE` images through the encoder per step. We therefore keep a **feature bank**: the most recent
embedding of each support image. Images in the current batch use their live embedding (gradients flow through
them), and all the others come from the bank (no gradients). The bank starts as the frozen embeddings, which equal the
adapted ones at initialisation because B = 0.

**Training recipe (paper):** 80 epochs, AdamW, lr 1e-4 (LoRA) and 1e-3 (tau_v), weight decay 1e-4, random resized
crops + horizontal flips, final checkpoint (no validation set).

In [ ]:
mean_std = [t for t in preprocess.transforms if isinstance(t, Tr.Normalize)][0]   # BiomedCLIP's pixel normalisation
train_tf = Tr.Compose([
    Tr.RandomResizedCrop(224, scale=(0.5, 1.0), interpolation=Tr.InterpolationMode.BICUBIC),  # random zoom + crop
    Tr.RandomHorizontalFlip(),                       # mirror left-right half of the time
    Tr.ToTensor(),                                   # PIL image -> [3, 224, 224] tensor in [0, 1]
    mean_std,                                        # same normalisation as the pretrained model expects
])  # crop scale range is not specified in the paper; (0.5, 1) keeps most of each lesion in view

def train_visual_branch(sup_idx, sup_y, T, seed, epochs=EPOCHS, verbose=True):
    # sup_idx: pool indices of the support set; sup_y: their labels 0..K-1 (local to T); T: [K, 512] text anchors
    # Trains the LoRA weights and tau_v with the leave-one-out loss. Returns tau_v and the loss history.
    set_seed(seed)                                   # optimisation seed (the paper uses 3 per support draw)
    for m in lora_modules: m.reset()                 # start from the frozen encoder every time
    Kc = T.shape[0]                                  # number of classes in this task
    imgs = [get_pool(i) for i in sup_idx]            # keep the support images in memory (PIL)
    y = torch.as_tensor(sup_y, dtype=torch.long, device=device)        # [N] labels
    N = len(imgs)
    onehot = F.one_hot(y, Kc).float()                # [N, Kc]: row i has a 1 at column y_i

    log_tau_v = nn.Parameter(torch.tensor(math.log(tau0), device=device))   # tau_v, starts at tau0 (log for positivity)
    opt = torch.optim.AdamW([{"params": lora_params, "lr": LR_LORA},
                             {"params": [log_tau_v], "lr": LR_SCALE}], weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)   # scales the loss so 16-bit gradients don't underflow

    bank = encode(get_pool, sup_idx, adapted=True).clone()   # [N, 512] feature bank (= frozen features right now)
    history = []
    for epoch in range(epochs):
        perm = torch.randperm(N, device=device)      # new random order of support images every epoch
        tot = 0.0
        for s in range(0, N, BATCH_SIZE):
            b = perm[s:s + BATCH_SIZE]               # indices (into the support set) of this batch
            x = torch.stack([train_tf(imgs[j]) for j in b.tolist()]).to(device)   # augmented images [B, 3, 224, 224]
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                f = model.encode_image(x)            # adapted embeddings F^S_v(x_i), gradients flow into LoRA
            f = F.normalize(f.float(), dim=-1)       # [B, 512], unit length

            feats = bank.index_put((b,), f)          # [N, 512]: bank, with this batch's rows replaced by live features
            S = T + onehot.T @ feats                 # [Kc, 512]: t_c + sum of class-c support features (Eq. 2, unnormalised)
            own = F.one_hot(y[b], Kc).float().unsqueeze(-1) * f.unsqueeze(1)   # [B, Kc, 512]: f_i at its own class only
            P = F.normalize(S.unsqueeze(0) - own, dim=-1)   # [B, Kc, 512]: leave-one-out prototypes p_{c,-i}
            logits = log_tau_v.exp() * torch.einsum("bd,bcd->bc", f, P)   # [B, Kc]: tau_v * <f_i, p_{c,-i}>
            loss = F.cross_entropy(logits, y[b])     # Eq. 3

            opt.zero_grad(set_to_none=True)          # clear old gradients
            scaler.scale(loss).backward()            # backward pass: gradients for LoRA and tau_v
            scaler.step(opt)                         # AdamW update
            scaler.update()                          # adjust the loss scale for next step
            bank[b] = f.detach()                     # refresh the bank with the newest features
            tot += loss.item() * len(b)
        history.append(tot / N)                      # average loss this epoch
        if verbose and (epoch % 10 == 0 or epoch == epochs - 1):
            print(f"epoch {epoch + 1:3d}/{epochs} | loss {history[-1]:.4f} | tau_v {log_tau_v.exp().item():.2f}")
    return log_tau_v.exp().item(), history

t0 = time.time()
sup_y = pool_labels[sup_idx]                         # labels of the support draw from section 4
tau_v, hist = train_visual_branch(sup_idx, sup_y, T_anchor, seed=SEED)
print(f"trained in {time.time() - t0:.0f}s")
plt.figure(figsize=(5, 3)); plt.plot(hist); plt.xlabel("epoch"); plt.ylabel("leave-one-out loss"); plt.title("training"); plt.show()

## 9. Inference: visual branch, support geometry, and the PARA prediction

### 9a. Visual branch (Eq. 4)
At test time, prototypes use **all** support images (no leave-one-out) encoded with the adapted encoder and the
deterministic preprocessing (no augmentation):  $z^S_c(x) = \tau_v \langle F^S_v(x), p^S_c \rangle$.

### 9b. Frozen support geometry (Eq. 5-7)
Using the **frozen** support embeddings $u_i = F^0_v(x_i)$ (so it does not depend on training randomness):

- $h_c = \sum_{i \in S_c} u_i$ (the class resultant vector), $d_c = h_c / \lVert h_c \rVert$ (its direction)
- **anchor discrepancy** $\delta_c = 1 - \langle d_c, t_c \rangle \in [0, 2]$: 0 if the images point exactly where the text points
- **support agreement** $\phi_c = \lVert h_c \rVert^2 / n_c \in [0, n_c]$: unit vectors that agree add up
  (|h| = n, phi = n), vectors that disagree cancel (phi small). Equals 1 with one shot.

### 9c. Residual step (Eq. 10)
$$\eta(S) = \frac{\lambda}{C} \sum_c \frac{\delta_c \phi_c}{1 + \delta_c \phi_c} \in [0, \lambda)$$
The product delta x phi is a soft "AND", a / (1 + a) squashes each class's vote into [0, 1), and averaging over
classes makes it comparable across datasets.

### 9d. Final prediction (Eq. 9)
Standardise each branch's scores across classes, then move from the anchor towards the visual branch by eta:
$$\hat z_c(x) = \bar z^0_c(x) + \eta(S)\,\big(\bar z^S_c(x) - \bar z^0_c(x)\big), \qquad \hat y = \arg\max_c \hat z_c(x)$$

In [ ]:
@torch.no_grad()
def visual_branch_logits(sup_idx, sup_y, T, tau_v, f_test_adapted):
    # Eq. 2 with all support images + Eq. 4
    Kc = T.shape[0]
    fs = encode(get_pool, sup_idx, adapted=True)              # [N, 512] adapted support embeddings, no augmentation
    onehot = F.one_hot(torch.as_tensor(sup_y, dtype=torch.long, device=device), Kc).float()   # [N, Kc]
    protos = F.normalize(T + onehot.T @ fs, dim=-1)           # [Kc, 512] prototypes p^S_c
    return tau_v * f_test_adapted @ protos.T                  # [N_test, Kc] visual scores z^S

@torch.no_grad()
def support_geometry(sup_idx, sup_y, T, lam=LAMBDA):
    # Eq. 5-7 and 10, computed from FROZEN support embeddings u_i
    Kc = T.shape[0]
    u = encode(get_pool, sup_idx, adapted=False)              # [N, 512] u_i = F0_v(x_i)
    onehot = F.one_hot(torch.as_tensor(sup_y, dtype=torch.long, device=device), Kc).float()
    h = onehot.T @ u                                          # [Kc, 512] h_c = sum of class-c support vectors
    n = onehot.sum(0)                                         # [Kc] n_c = number of support images per class
    d = F.normalize(h, dim=-1)                                # [Kc, 512] visual direction d_c
    delta = 1 - (d * T).sum(-1)                               # [Kc] anchor discrepancy, 1 - cos(d_c, t_c)
    phi = h.norm(dim=-1) ** 2 / n                             # [Kc] support agreement ||h_c||^2 / n_c
    a = delta * phi                                           # [Kc] soft conjunction
    eta = lam * (a / (1 + a)).mean().item()                   # Eq. 10: bounded, averaged over classes
    return eta, delta, phi, d

def para_logits(z0, zS, eta):
    # Eq. 9: anchor + eta * (visual - anchor), both standardised across classes
    z0b, zSb = standardize(z0), standardize(zS)
    return z0b + eta * (zSb - z0b)

fS_test = encode(get_test, test_idx, adapted=True)            # [N_test, 512] adapted test embeddings F^S_v(x)
zS_test = visual_branch_logits(sup_idx, sup_y, T_anchor, tau_v, fS_test)   # visual branch scores
eta, delta, phi, _ = support_geometry(sup_idx, sup_y, T_anchor)            # residual step from frozen geometry
zP_test = para_logits(z0_test, zS_test, eta)                 # final PARA scores

print(f"{'class':<48} {'delta':>7} {'phi':>7} {'delta*phi':>10}")
for c in range(C):
    print(f"{classnames[c][:48]:<48} {delta[c]:7.3f} {phi[c]:7.2f} {(delta[c] * phi[c]):10.3f}")
print(f"\neta(S) = {eta:.3f}   (upper bound lambda = {LAMBDA})")

## 10. Results and free ablations

Because PARA only mixes two sets of scores, two of the paper's ablations (Table 4) cost nothing to compute:
- **w/o prompt-anchored residual**: use the visual branch alone (eta = 1)
- **w/o support geometry**: fixed step eta = 0.5 instead of eta(S)

(The third ablation, *w/o prototype anchoring*, drops `t_c` from Eq. 2 and needs retraining, so we skip it.)

Expect PARA to be at least as good as both branches on average. When the text anchor is weak and the visual branch is
strong (the paper's CHMNIST failure case), PARA can land between them.

In [ ]:
results = {
    "Zero-shot (anchor)":     metrics(z0_test, y_test),                         # eta = 0
    "Visual branch only":     metrics(zS_test, y_test),                         # eta = 1
    "Fixed eta = 0.5":        metrics(para_logits(z0_test, zS_test, 0.5), y_test),
    f"PARA (eta = {eta:.2f})": metrics(zP_test, y_test),
}
for k, (a, f) in results.items():
    print(f"{k:<22} | Acc {a:6.2f} | Macro-F1 {f:6.2f}")

# Sweep eta from 0 to 1 to see where eta(S) lands on the curve
etas = np.linspace(0, 1, 21)
sweep = np.array([metrics(para_logits(z0_test, zS_test, e), y_test) for e in etas])   # [21, 2]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(etas, sweep[:, 0], label="Accuracy"); ax.plot(etas, sweep[:, 1], label="Macro-F1")
ax.axvline(eta, color="k", ls="--", label=f"eta(S) = {eta:.2f}")   # the step PARA picked, without any labels
ax.set_xlabel("residual step eta (0 = anchor, 1 = visual)"); ax.set_ylabel("%"); ax.legend(); ax.set_title(DATASET)
plt.tight_layout(); plt.show()

## 11. (Optional) Repeated-support protocol: mean and CVaR20

The paper's second contribution is *how to evaluate*. One support draw tells you little, because a different draw can
give a very different score. Their protocol:

1. Draw **D = 20** support sets per dataset and shot level, shared by all methods.
2. For each draw, run **3 optimisation seeds** and average them. This removes optimiser noise, so the remaining
   spread comes from *which images were drawn*.
3. Report the **mean** over draws, and **CVaR20** = average of the worst 20% of draws (the 4 lowest of 20).
   CVaR20 answers "how bad is it if I'm unlucky with my labelled examples?"

Full scale (20 draws x 3 seeds x 5 shot levels) is far too slow for Colab, so the defaults below are small.
CVaR20 then averages the worst `ceil(0.2 * D)` draws.

In [ ]:
RUN_PROTOCOL = True        # set False to skip this section
N_DRAWS = 5                # paper: 20
N_OPT_SEEDS = 1            # paper: 3
PROTOCOL_SHOTS = [SHOTS]   # paper: [1, 2, 4, 8, 16]

def cvar20(values):
    # mean of the worst 20% of support draws (at least one)
    v = np.sort(np.asarray(values))                         # ascending: worst first
    return v[:max(1, math.ceil(0.2 * len(v)))].mean()

if RUN_PROTOCOL:
    methods = ["Zero-shot", "Visual only", "Fixed eta 0.5", "PARA"]
    table = {}                                               # (shots, method) -> list of per-draw (acc, f1)
    for k in PROTOCOL_SHOTS:
        for d in range(N_DRAWS):
            s_idx = draw_support(k, draw_seed=d)             # same draw d for every method
            s_y = pool_labels[s_idx]
            eta_d = support_geometry(s_idx, s_y, T_anchor)[0]   # eta depends only on the draw, not the seed
            per_seed = {m: [] for m in methods}
            for s in range(N_OPT_SEEDS):
                tv, _ = train_visual_branch(s_idx, s_y, T_anchor, seed=s, verbose=False)
                zS = visual_branch_logits(s_idx, s_y, T_anchor, tv, encode(get_test, test_idx, adapted=True))
                per_seed["Zero-shot"].append(metrics(z0_test, y_test))
                per_seed["Visual only"].append(metrics(zS, y_test))
                per_seed["Fixed eta 0.5"].append(metrics(para_logits(z0_test, zS, 0.5), y_test))
                per_seed["PARA"].append(metrics(para_logits(z0_test, zS, eta_d), y_test))
            for m in methods:                                # Eq. A1: average the seeds within this draw
                table.setdefault((k, m), []).append(np.mean(per_seed[m], axis=0))
            print(f"K={k} draw {d + 1}/{N_DRAWS} | eta {eta_d:.3f} | PARA Acc {table[(k, 'PARA')][-1][0]:.2f}")

    print(f"\n{'K':>3} {'method':<14} {'Acc mean':>9} {'Acc SD':>7} {'Acc CVaR20':>11} {'F1 mean':>8} {'F1 CVaR20':>10}")
    for (k, m), vals in table.items():
        v = np.array(vals)                                   # [D, 2] per-draw (acc, f1)
        print(f"{k:>3} {m:<14} {v[:, 0].mean():9.2f} {v[:, 0].std():7.2f} {cvar20(v[:, 0]):11.2f}"
              f" {v[:, 1].mean():8.2f} {cvar20(v[:, 1]):10.2f}")    # SD = population SD over draws (Eq. A2)

## 12. Base-to-novel: residual transport (Eq. 11-14)

Split the classes into **base** (with 16-shot support) and **novel** (no labelled images at all). Base test images
are classified among base classes and novel test images among novel classes (the standard protocol).

- **Base classes:** ordinary PARA trained on base support only.
- **Novel classes:** no support -> no visual branch, no `d_n`. Instead, borrow how image directions differ from text
  directions on base classes:
  1. Displacement for each base class: $\Delta_b = d_b - t_b$ (Eq. 11), in the frozen space.
  2. Weights from text similarity: $w_{nb} = \mathrm{softmax}_b(\beta \langle t_n, t_b \rangle)$ (Eq. 12). Higher beta =
     only the most similar base classes count.
  3. Shifted anchor: $t'_n = \mathrm{normalize}(t_n + \gamma \sum_b w_{nb} \Delta_b)$ (Eq. 13), gamma = transfer size.
  4. Score with the frozen encoder: $z'_n(x) = \tau_0 \langle F^0_v(x), t'_n \rangle$ (Eq. 14).

Intuition: if images of "melanoma" sit in a different direction than the word embeddings suggest, a similar novel class
probably has a similar text-image gap, so we nudge its anchor the same way.

We use the first half of the classes as base (the paper uses BiomedCoOp's splits).

In [ ]:
n_base = math.ceil(C / 2)                                     # e.g. 4 of 7 classes
base_cls, novel_cls = list(range(n_base)), list(range(n_base, C))
print("base :", [classnames[c] for c in base_cls])
print("novel:", [classnames[c] for c in novel_cls])
T_base, T_novel = T_anchor[base_cls], T_anchor[novel_cls]     # [B, 512], [Nn, 512] text anchors

# --- Base classes: PARA trained on base support only ---
b_idx = draw_support(16, draw_seed=0, classes=base_cls)       # 16-shot support from base classes only
b_y = pool_labels[b_idx]                                      # global labels 0..n_base-1 (base classes come first)
tv_b, _ = train_visual_branch(b_idx, b_y, T_base, seed=SEED, verbose=False)
eta_b, _, _, d_base = support_geometry(b_idx, b_y, T_base)    # eta and base visual directions d_b (frozen space)

mb = np.isin(y_test, base_cls)                                # test images whose true class is a base class
yb = y_test[mb]                                               # their labels (already 0..n_base-1)
z0_b = tau0 * f0_test[mb] @ T_base.T                          # anchor scores among base classes
zS_b = visual_branch_logits(b_idx, b_y, T_base, tv_b, encode(get_test, test_idx[mb], adapted=True))
base_zs, base_para = metrics(z0_b, yb), metrics(para_logits(z0_b, zS_b, eta_b), yb)

# --- Novel classes: residual transport ---
Delta = d_base - T_base                                       # [B, 512] Eq. 11: visual - text displacement
w = torch.softmax(BETA * T_novel @ T_base.T, dim=-1)          # [Nn, B]  Eq. 12: semantic transfer weights
T_novel_new = F.normalize(T_novel + GAMMA * w @ Delta, dim=-1)   # [Nn, 512] Eq. 13: shifted novel anchors

mn = np.isin(y_test, novel_cls)                               # test images of novel classes
yn = y_test[mn] - n_base                                      # relabel to 0..Nn-1
novel_zs = metrics(tau0 * f0_test[mn] @ T_novel.T, yn)        # zero-shot novel
novel_para = metrics(tau0 * f0_test[mn] @ T_novel_new.T, yn)  # Eq. 14: transported anchors, frozen encoder

hm = lambda b, n: 2 * b * n / (b + n)                         # harmonic mean of base and novel
print(f"eta(base support) = {eta_b:.3f}")
print("transfer weights w_nb (rows: novel, cols: base):\n", np.round(w.cpu().numpy(), 2))
print(f"\n{'':<12} {'Base Acc':>9} {'Novel Acc':>10} {'HM':>7} | {'Base F1':>8} {'Novel F1':>9} {'HM':>7}")
for name, b, n in [("Zero-shot", base_zs, novel_zs), ("PARA", base_para, novel_para)]:
    print(f"{name:<12} {b[0]:9.2f} {n[0]:10.2f} {hm(b[0], n[0]):7.2f} | {b[1]:8.2f} {n[1]:9.2f} {hm(b[1], n[1]):7.2f}")

## 13. Things to try

- `SHOTS = 1` vs `16`: watch eta(S) grow with K (phi grows with agreeing examples), and the gap between PARA and the
  visual branch change.
- Another dataset (`DATASET = "octmnist"`, `"pathmnist"`, `"bloodmnist"`): on datasets where zero-shot is very weak,
  the visual branch can beat PARA (the paper's CHMNIST failure analysis).
- `LAMBDA`: the paper fixes 0.9 everywhere with no tuning; the eta-sweep plot in section 10 shows how much is lost or
  gained by that choice on your data.
- Raise `N_DRAWS` in section 11 and compare the **CVaR20** of "Visual only" against PARA: the paper's main claim is that
  anchoring helps the worst draws most.